# 🔬 Cancer Detection — Training Pipeline

**Before running anything:**
1. Go to **Runtime → Change runtime type → T4 GPU**
2. Click **Connect** (top right)
3. Run cells top to bottom — do not skip any cell

---
| Component | Detail |
|-----------|--------|
| Model | MobileNetV2 (ImageNet backbone, frozen) |
| Dataset | Breast Histopathology Images (Kaggle) |
| Task | Binary classification: cancer vs non-cancer |
| Output | `model.h5`, `loss.png`, `accuracy.png` on Drive |

## Cell 1 — GPU Check

In [ ]:
import subprocess

result = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
if result.returncode == 0:
    print("✅ GPU detected")
    for line in result.stdout.split("\n")[8:11]:
        print(" ", line)
else:
    raise RuntimeError(
        "❌ No GPU found.\n"
        "Go to Runtime → Change runtime type → T4 GPU, then reconnect."
    )

## Cell 2 — Install Dependencies

In [ ]:
import subprocess, sys

packages = ["kaggle", "tensorflow", "opencv-python-headless", "matplotlib", "pillow"]
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + packages)
print("✅ Dependencies installed")

## Cell 3 — Mount Google Drive

In [ ]:
from google.colab import drive
from pathlib import Path

drive.mount("/content/drive")

DRIVE_ROOT    = Path("/content/drive/MyDrive/CancerDetection")
DRIVE_DATASET = DRIVE_ROOT / "dataset"
DRIVE_OUTPUT  = DRIVE_ROOT / "outputs"

for folder in [
    DRIVE_DATASET / "cancer",
    DRIVE_DATASET / "non-cancer",
    DRIVE_OUTPUT,
]:
    folder.mkdir(parents=True, exist_ok=True)

print(f"✅ Drive mounted")
print(f"   Project root : {DRIVE_ROOT}")
print(f"   Dataset path : {DRIVE_DATASET}")
print(f"   Output path  : {DRIVE_OUTPUT}")

## Cell 4 — Kaggle Authentication

Get your token at [kaggle.com/settings](https://www.kaggle.com/settings) → **API → Create New Token**.  
This downloads `kaggle.json`. Run the cell and upload it when prompted.

In [ ]:
from google.colab import files
import os, json, stat
from pathlib import Path

print("📂 Upload your kaggle.json when the file picker appears ↓")
uploaded = files.upload()

kaggle_dir = Path("/root/.kaggle")
kaggle_dir.mkdir(exist_ok=True)
kaggle_json_path = kaggle_dir / "kaggle.json"

for filename, content in uploaded.items():
    kaggle_json_path.write_bytes(content)

# Validate JSON structure
creds = json.loads(kaggle_json_path.read_text())
assert "username" in creds and "key" in creds, \
    "❌ kaggle.json must contain both 'username' and 'key' fields"

# Required permission (Kaggle CLI refuses if too permissive)
os.chmod(kaggle_json_path, stat.S_IRUSR | stat.S_IWUSR)

print(f"✅ Kaggle credentials verified for user: {creds['username']}")

## Cell 5 — Download Dataset from Kaggle

**Dataset**: [Breast Histopathology Images](https://www.kaggle.com/datasets/paultimothymooney/breast-histopathology-images)  
~2.7 GB unzipped · 162,000 image patches · Classes: `0` = non-cancer, `1` = cancer  
⏱️ *Expected download time: 5–10 minutes on Colab*

In [ ]:
import subprocess
from pathlib import Path

RAW_DOWNLOAD_DIR = Path("/content/raw_kaggle")
RAW_DOWNLOAD_DIR.mkdir(exist_ok=True)

print("⬇️  Downloading dataset... (5–10 minutes)")

result = subprocess.run(
    [
        "kaggle", "datasets", "download",
        "-d", "paultimothymooney/breast-histopathology-images",
        "-p", str(RAW_DOWNLOAD_DIR),
        "--unzip",
    ],
    capture_output=True,
    text=True,
)

if result.returncode != 0:
    print("STDERR:", result.stderr)
    raise RuntimeError(
        "❌ Download failed. Check:\n"
        "  1. Kaggle credentials (re-run Cell 4)\n"
        "  2. Dataset slug is correct\n"
        "  3. You have accepted the dataset terms on Kaggle"
    )

all_images = list(RAW_DOWNLOAD_DIR.rglob("*.png"))
print(f"✅ Download complete")
print(f"   Raw images found: {len(all_images):,}")

## Cell 6 — Organize into `cancer/` and `non-cancer/`

> **`MAX_PER_CLASS`** controls training size/speed trade-off:
> - `5_000` → ~10 min/epoch, good for a first run
> - `20_000` → ~40 min/epoch, better accuracy
> - `50_000+` → production quality, several hours

In [ ]:
import shutil
from pathlib import Path

MAX_PER_CLASS = 5_000  # ← change this to use more data

cancer_dir    = DRIVE_DATASET / "cancer"
noncancer_dir = DRIVE_DATASET / "non-cancer"

cancer_existing    = len(list(cancer_dir.iterdir()))
noncancer_existing = len(list(noncancer_dir.iterdir()))

if cancer_existing >= MAX_PER_CLASS and noncancer_existing >= MAX_PER_CLASS:
    print(f"✅ Dataset already organized (skipping copy):")
    print(f"   cancer/     → {cancer_existing:,} images")
    print(f"   non-cancer/ → {noncancer_existing:,} images")
else:
    print(f"📦 Organizing up to {MAX_PER_CLASS:,} images per class...")
    cancer_count    = cancer_existing
    noncancer_count = noncancer_existing

    for img_path in RAW_DOWNLOAD_DIR.rglob("*.png"):
        label = img_path.parent.name  # "0" or "1"

        if label == "1" and cancer_count < MAX_PER_CLASS:
            shutil.copy2(img_path, cancer_dir / img_path.name)
            cancer_count += 1
        elif label == "0" and noncancer_count < MAX_PER_CLASS:
            shutil.copy2(img_path, noncancer_dir / img_path.name)
            noncancer_count += 1

        if cancer_count >= MAX_PER_CLASS and noncancer_count >= MAX_PER_CLASS:
            break

    print(f"✅ Organization complete:")
    print(f"   cancer/     → {cancer_count:,} images")
    print(f"   non-cancer/ → {noncancer_count:,} images")

# Hard assertions before proceeding
assert len(list(cancer_dir.iterdir())) > 0,    "❌ cancer/ is still empty"
assert len(list(noncancer_dir.iterdir())) > 0, "❌ non-cancer/ is still empty"
print("✅ Dataset structure verified")

## Cell 7 — Set Working Directory & Verify Project Files

In [ ]:
import os, sys
from pathlib import Path

os.chdir(str(DRIVE_ROOT))
sys.path.insert(0, str(DRIVE_ROOT))

print(f"✅ Working directory: {os.getcwd()}")

required_files = ["model/utils.py", "model/train.py"]
for f in required_files:
    exists = Path(f).exists()
    status = "✅" if exists else "❌"
    print(f"   {status} {f}")
    assert exists, (
        f"Missing {f}. Copy your project to {DRIVE_ROOT}/ "
        "before running this notebook."
    )

print("✅ All source files found")

## Cell 15 — Improved Augmentation

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 15 — IMPROVED AUGMENTATION
# Replace existing ImageDataGenerator with this
# ─────────────────────────────────────────────────────────────────────────────
from tensorflow.keras.preprocessing.image import ImageDataGenerator

IMG_SIZE   = 96     # keep existing size
BATCH_SIZE = 32     # keep existing batch size

train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    width_shift_range=0.1,
    height_shift_range=0.1,
    horizontal_flip=True,
    vertical_flip=True,
    zoom_range=0.15,
    brightness_range=[0.8, 1.2],    # NEW: brightness variation
    channel_shift_range=10.0,        # NEW: slight color noise
    fill_mode='nearest',
    validation_split=0.2
)

# Validation: ONLY rescale, no augmentation
val_datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2
)

train_gen = train_datagen.flow_from_directory(
    str(DRIVE_DATASET),
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='binary',
    subset='training',
    seed=42
)

val_gen = val_datagen.flow_from_directory(
    str(DRIVE_DATASET),
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='binary',
    subset='validation',
    seed=42
)

print(f"✅ Augmented train generator: {train_gen.samples:,} images")
print(f"✅ Clean val generator      : {val_gen.samples:,} images")\n

## Cell 8 — Data Pipeline Smoke Test

In [ ]:
from model.utils import create_image_generators

print("🔍 Validating data pipeline...")
train_gen, val_gen = create_image_generators(
    dataset_path=str(DRIVE_DATASET),
    image_size=(224, 224),
    batch_size=32,
)

print(f"\n✅ Generators created")
print(f"   Train samples      : {train_gen.samples:,}")
print(f"   Validation samples : {val_gen.samples:,}")
print(f"   Class indices      : {train_gen.class_indices}")

batch_x, batch_y = next(train_gen)
print(f"\n   Batch image shape  : {batch_x.shape}")  # (32, 224, 224, 3)
print(f"   Batch label shape  : {batch_y.shape}")   # (32,)
print(f"   Pixel value range  : [{batch_x.min():.3f}, {batch_x.max():.3f}]")  # [0, 1]

assert batch_x.shape == (32, 224, 224, 3), f"Unexpected shape: {batch_x.shape}"
assert batch_x.max() <= 1.0,              "Normalization failed — pixel values > 1.0"
print("\n✅ Smoke test PASSED")

## Cell 9 — Build Model & Verify Architecture

In [ ]:
from model.train import build_model, get_parameter_counts

print("🔍 Building MobileNetV2 classifier...")
model = build_model(input_shape=(224, 224, 3))

counts = get_parameter_counts(model)
print(f"\n✅ Model built: {model.name}")
print(f"   Trainable params     : {counts['trainable']:,}")
print(f"   Non-trainable params : {counts['non_trainable']:,}")
print(f"   Total params         : {counts['total']:,}")

# Verify backbone is frozen
for layer in model.layers:
    if "mobilenet" in layer.name.lower():
        assert not layer.trainable, "❌ MobileNetV2 backbone must be frozen"
print("   Backbone frozen      : ✅")

model.summary()

## Cell 14 — Class Weight Computation

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 14 — CLASS WEIGHT COMPUTATION
# Run this before model.fit() in Cell 10/11
# ─────────────────────────────────────────────────────────────────────────────
import numpy as np
from sklearn.utils.class_weight import compute_class_weight

# Get all labels from training generator
train_labels = train_gen.classes

# Compute balanced class weights
class_weights_array = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(train_labels),
    y=train_labels
)

class_weights = {
    0: class_weights_array[0],
    1: class_weights_array[1]
}

print("=" * 45)
print("CLASS WEIGHT ANALYSIS")
print("=" * 45)
total = len(train_labels)
cancer_count = np.sum(train_labels == 1)
noncancer_count = np.sum(train_labels == 0)
print(f"Total training images : {total:,}")
print(f"Cancer (class 1)      : {cancer_count:,} ({cancer_count/total*100:.1f}%)")
print(f"Non-Cancer (class 0)  : {noncancer_count:,} ({noncancer_count/total*100:.1f}%)")
print(f"\nComputed class weights:")
print(f"  class 0 (Non-Cancer): {class_weights[0]:.4f}")
print(f"  class 1 (Cancer)    : {class_weights[1]:.4f}")
print("=" * 45)
print("✅ Pass class_weights to model.fit() in Cell 10/11")\n

## Cell 10 — Train Model

Callbacks active:
- **EarlyStopping** — stops if `val_loss` doesn't improve for 3 epochs
- **ModelCheckpoint** — saves best model to `best_model.h5`
- **ReduceLROnPlateau** — halves learning rate on plateau

In [ ]:
import tensorflow as tf

EPOCHS = 10  # EarlyStopping may stop this earlier

callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=3,
        restore_best_weights=True,
        verbose=1,
    ),
    tf.keras.callbacks.ModelCheckpoint(
        filepath=str(DRIVE_OUTPUT / "best_model.h5"),
        monitor="val_loss",
        save_best_only=True,
        verbose=1,
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=2,
        min_lr=1e-6,
        verbose=1,
    ),
]

print(f"🚀 Training for up to {EPOCHS} epochs...")
history = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=EPOCHS,
    callbacks=callbacks,
)
print("\n

## Cell 11 — Save Artifacts to Drive

In [ ]:
from model.train import save_training_artifacts

save_training_artifacts(model, history, DRIVE_OUTPUT)

print("✅ Artifacts saved to Drive:")
for f in sorted(DRIVE_OUTPUT.iterdir()):
    size_kb = f.stat().st_size / 1024
    print(f"   {f.name:25s}  {size_kb:>8.1f} KB")

## Cell 12 — Validation Summary & Plots

In [ ]:
import matplotlib.pyplot as plt

h = history.history
epochs_ran = len(h["loss"])

print("=" * 52)
print("  TRAINING RESULTS SUMMARY")
print("=" * 52)
print(f"  Epochs completed      : {epochs_ran}")
print(f"  Final train loss      : {h['loss'][-1]:.4f}")
print(f"  Final val loss        : {h['val_loss'][-1]:.4f}")
print(f"  Final train accuracy  : {h['accuracy'][-1]:.4f}")
print(f"  Final val accuracy    : {h['val_accuracy'][-1]:.4f}")
print(f"  Final precision       : {h['precision'][-1]:.4f}")
print(f"  Final recall          : {h['recall'][-1]:.4f}")
print("=" * 52)

# --- Checks ---
loss_ok      = h["loss"][0] > h["loss"][-1]
overfit_ratio = h["val_loss"][-1] / max(h["loss"][-1], 1e-8)
accuracy_ok  = h["val_accuracy"][-1] > 0.60

print(f"\n  Loss decreasing       : {'✅' if loss_ok      else '❌ FAIL'}")
print(f"  Val/Train loss ratio  : {overfit_ratio:.2f}  {'✅ OK' if overfit_ratio < 1.5 else '⚠️  Possible overfitting'}")
print(f"  Val accuracy > 60%    : {'✅' if accuracy_ok  else '⚠️  Low — check class balance'}")

# --- Plots ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(h["loss"],     label="Train Loss",        linewidth=2)
axes[0].plot(h["val_loss"], label="Validation Loss",   linewidth=2, linestyle="--")
axes[0].set_title("Loss Curve (Gradient Descent Behaviour)", fontsize=13)
axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("Loss")
axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(h["accuracy"],     label="Train Accuracy",      linewidth=2)
axes[1].plot(h["val_accuracy"], label="Validation Accuracy", linewidth=2, linestyle="--")
axes[1].set_title("Accuracy Curve", fontsize=13)
axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("Accuracy")
axes[1].legend(); axes[1].grid(alpha=0.3)

plt.suptitle("Cancer Detection — MobileNetV2 Training", fontsize=15, fontweight="bold")
plt.tight_layout()
plt.show()
print("\n✅ All validation checks complete")

## Cell 16 — Compute Optimal Decision Threshold

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 16 — COMPUTE OPTIMAL DECISION THRESHOLD
# Run after fine-tuning (Cell 13) is complete
# ─────────────────────────────────────────────────────────────────────────────
import numpy as np
import json
from sklearn.metrics import roc_curve, f1_score

print("Computing optimal threshold from validation set...")

# Load finetuned model
finetuned_path = str(DRIVE_OUTPUT / "best_model_finetuned.h5")
threshold_model = tf.keras.models.load_model(finetuned_path)

# Get predictions on full validation set
val_gen.reset()
val_preds = threshold_model.predict(val_gen, verbose=1)
val_labels = val_gen.classes[:len(val_preds)]

# Method 1: Best F1 Score threshold
thresholds = np.arange(0.3, 0.85, 0.01)
f1_scores  = []
for t in thresholds:
    preds = (val_preds.flatten() > t).astype(int)
    f1_scores.append(f1_score(val_labels, preds))

best_f1_threshold = thresholds[np.argmax(f1_scores)]

# Method 2: Youden's J (best for medical — balances sensitivity/specificity)
fpr, tpr, roc_thresholds = roc_curve(val_labels, val_preds)
youden_j = tpr - fpr
best_youden_threshold = float(roc_thresholds[np.argmax(youden_j)])

print("=" * 50)
print("THRESHOLD OPTIMIZATION RESULTS")
print("=" * 50)
print(f"Best F1 threshold       : {best_f1_threshold:.2f}")
print(f"Best Youden's J thresh  : {best_youden_threshold:.2f}")
print(f"\nRecommended (Youden's J): {best_youden_threshold:.2f}")
print("(Youden's J preferred for cancer detection —")
print(" minimizes missed cancers)")
print("=" * 50)

# Save to Drive alongside model
threshold_data = {
    'threshold': best_youden_threshold,
    'f1_threshold': float(best_f1_threshold),
    'method': "Youden's J statistic",
    'note': 'Use threshold value in app.py prediction logic'
}

threshold_path = DRIVE_OUTPUT / "model_threshold.json"
with open(threshold_path, 'w') as f:
    json.dump(threshold_data, f, indent=2)

print(f"\n✅ Threshold saved to: {threshold_path}")
print(f"   Update THRESHOLD in app.py to: {best_youden_threshold:.2f}")\n

## Cell 17 — Final Training Summary

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 17 — FINAL TRAINING SUMMARY
# ─────────────────────────────────────────────────────────────────────────────
print("=" * 55)
print("  FINAL TRAINING SUMMARY")
print("=" * 55)
print(f"\n  Phase 1 val accuracy   : {phase1_val_acc*100:.2f}%")
print(f"  Phase 2 val accuracy   : {phase2_val_acc*100:.2f}%")
delta = (phase2_val_acc - phase1_val_acc) * 100
print(f"  Improvement            : +{delta:.2f}%")
print(f"\n  Optimal threshold      : {best_youden_threshold:.2f}")
print(f"\n  Files saved to Drive:")
for f in sorted(DRIVE_OUTPUT.iterdir()):
    size_kb = f.stat().st_size / 1024
    print(f"    {f.name:30s} {size_kb:>8.1f} KB")
print("\n  Next steps:")
print("  1. Copy best_model_finetuned.h5 to your app folder")
print("  2. Copy model_threshold.json to your app folder")
print(f"  3. Set THRESHOLD = {best_youden_threshold:.2f} in app.py")
print("  4. Run: streamlit run app.py")
print("=" * 55)\n